### **NutriFit: Personalized Diet Recommendation System using Machine Learning and Content-Based Filtering **

Content-based recommendation - Personalized recommendatiob based on user profile and its ability to adapt to user preferences

In [1]:
# Uploading dataset to collab

from google.colab import files
uploaded = files.upload()


Saving Dietdataset.csv to Dietdataset.csv


### **Load Required Libraries**

In [2]:
# for handling the dataset
import pandas as pd
import numpy as np
# for performing stemming
import nltk
from nltk.stem.porter import PorterStemmer
# for creating vectors and finding their similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


### **Import the dataset**

In [3]:
#Importing dataset

diet=pd.read_csv("Dietdataset.csv")
diet.head()

,Unnamed: 0,age,weight(kg),height(m),gender,BMI,Disease,calories_to_maintain_weight,Vegan_Recipe,V_Protein(g),V_Carbs(g),V_Fat(g),Mediterranean_Recipe,M_Protein(g),M_Carbs(g),M_Fat(g),Cuisine_type
0,5498,12,35.930163,1.514615,M,15.662278,Hypothyroidism,1604.1363,Spiked Vegan Eggnog,46.36,84.52,42.61,Mediterranean Cucumber Tonic,0.25,12.01,0.33,mediterranean
1,5499,11,41.826932,1.506550,M,18.428461,Thyroid,1581.0048,Vegan Cucumber Tea Sandwiches,41.73,143.91,73.88,Loup de Mer (Mediterranean Seabass),306.14,43.10,26.55,american
2,5500,13,34.331143,1.485346,F,15.560849,Thyroid,1763.9820,Vegan Broccoli Soup Recipe,103.83,82.54,139.30,"Mediterranean Baked Trout with Olives, Fennel ...",187.14,42.31,87.94,asian
3,5502,10,39.840596,1.470897,F,18.414559,Thyroid,2356.9918,Baked Vegan Mac and Cheese Recipe,140.19,524.38,221.29,Mediterranean Salmon Risotto recipes,395.50,287.10,291.04,nordic
4,5503,12,38.554640,1.542641,F,16.201189,Thyroid,1477.1688,Vegan Nachos,73.05,266.38,166.64,Mediterranean Pita Melts,46.57,139.71,37.70,mediterranean


### **Data Preprocssing**

In [4]:
#Data Preprocssing
#Dropping unnecessary columns from dataset

diet = diet.drop(diet.columns[0], axis=1)
diet.drop("Cuisine_type",axis=1,inplace=True)
diet.head(3)

,age,weight(kg),height(m),gender,BMI,Disease,calories_to_maintain_weight,Vegan_Recipe,V_Protein(g),V_Carbs(g),V_Fat(g),Mediterranean_Recipe,M_Protein(g),M_Carbs(g),M_Fat(g)
0,12,35.930163,1.514615,M,15.662278,Hypothyroidism,1604.1363,Spiked Vegan Eggnog,46.36,84.52,42.61,Mediterranean Cucumber Tonic,0.25,12.01,0.33
1,11,41.826932,1.506550,M,18.428461,Thyroid,1581.0048,Vegan Cucumber Tea Sandwiches,41.73,143.91,73.88,Loup de Mer (Mediterranean Seabass),306.14,43.10,26.55
2,13,34.331143,1.485346,F,15.560849,Thyroid,1763.9820,Vegan Broccoli Soup Recipe,103.83,82.54,139.30,"Mediterranean Baked Trout with Olives, Fennel ...",187.14,42.31,87.94


In [5]:
try:

    # Clean up column names (remove leading/trailing spaces)
    diet.columns = diet.columns.str.strip()

    # Check if the columns exist and then apply rounding
    if 'weight(kg)' in diet.columns and 'height(m)' in diet.columns:
        diet['weight(kg)'] = diet['weight(kg)'].apply(lambda x: int(round(x)))
        diet['height(m)'] = diet['height(m)'].apply(lambda x: round(x, 1))
        diet['BMI'] = diet.apply(lambda row: round(row['weight(kg)'] / (row['height(m)'] ** 2), 1), axis=1)
except Exception as e:
    print("Error:", e)

In [6]:
#check null values

diet.isnull().sum()

age                            0
weight(kg)                     0
height(m)                      0
gender                         0
BMI                            0
Disease                        0
calories_to_maintain_weight    0
Vegan_Recipe                   0
V_Protein(g)                   0
V_Carbs(g)                     0
V_Fat(g)                       0
Mediterranean_Recipe           0
M_Protein(g)                   0
M_Carbs(g)                     0
M_Fat(g)                       0
dtype: int64

### **TF-IDF Vectorization and Cosine similarity**

*   Text Preprocessing:Cleaning prepares text which involves tasks like removing punctuation, converting text to lowercase, removing stopwords
*   Tokenization:Break down the cleaned text into individual words.
*   Vectorization:Method to convert each recipe description into a numerical vector.
*   Cosine similarity:Compute similarities between recipe vectors to find similar recipes.

In [8]:

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define a function to calculate BMI
def calculate_BMI(weight_kg, height_m):
    return weight_kg / (height_m ** 2)

# Define a function to recommend diets based on user input
def recommend_diet(age, weight_kg, height_m, gender, disease, diet_preference, diet_df):
    # Check if age and weight are within expected range
    if not (10 <= age <= 90) or not (weight_kg <= 98):
        raise ValueError("Input values are outside the expected range for diet recommendations.")

    # Calculate BMI
    BMI = calculate_BMI(weight_kg, height_m)

    # Filter diet data based on user's disease
    filtered_diet = diet_df[diet_df["Disease"] == disease]

    if filtered_diet.empty:
        raise ValueError("No suitable diet recommendations found for the given disease.")

    # Select recipe columns based on diet preference
    if diet_preference.lower() == "vegan":
        recipe_columns = ["Vegan_Recipe", "V_Protein(g)", "V_Carbs(g)", "V_Fat(g)"]
    elif diet_preference.lower() == "mediterranean":
        recipe_columns = ["Mediterranean_Recipe", "M_Protein(g)", "M_Carbs(g)", "M_Fat(g)"]
    else:
        raise ValueError("Invalid diet preference. Choose either 'vegan' or 'mediterranean'.")

    # Preprocess recipe text and vectorize using TF-IDF
    recipes_text = filtered_diet[recipe_columns[0]].fillna("").astype(str)

    vectorizer = TfidfVectorizer(stop_words='english')
    recipe_matrix = vectorizer.fit_transform(recipes_text)

    # Transform user preferences into TF-IDF vector
    user_preferences = f"{age} {weight_kg} {height_m} {gender} {BMI} {disease}"
    user_vector = vectorizer.transform([user_preferences])

    # Calculate cosine similarity between user vector and recipe matrix
    cosine_similarities = cosine_similarity(user_vector, recipe_matrix).flatten()

    # Get top recommendation based on highest similarity score
    top_index = cosine_similarities.argsort()[::-1][0]
    recommended_recipe = filtered_diet.iloc[top_index]

    return recommended_recipe, BMI



### **Test Cases**

In [ ]:
# Test cases

# Test case 1: Typical user inputs from dataset
age = 63
weight_kg = 97.61518559
height_m = 1.633168175
gender = "M"
disease = "Heart Disease"
diet_preference = "Vegan"  # Specify 'Vegan' or 'Mediterranean'

# Test case 2: Close match to dataset
# age = 12
# weight_kg = 34.33114267
# height_m = 1.485345504
# gender = "M"
# disease = "Thyroid"
# diet_preference = "Mediterranean"

# Test case 3: Unsupported disease
# age = 30
# weight_kg = 60
# height_m = 1.65
# gender = "F"
# disease = "Unknown Disease"
# diet_preference = "Vegan"

 #Test case 4: Extreme values (not in dataset)
#age = 100
#weight_kg = 200
#height_m = 1.80
#gender = "M"
#disease = "Heart Disease"
#diet_preference = "Mediterranean"

    # Call recommend_diet function with user input and diet dataset
try:
    recommendation, calculated_BMI = recommend_diet(age, weight_kg, height_m, gender, disease, diet_preference, diet)

    # Print diet recommendation and calculated BMI in a formatted table
    if recommendation is not None:
        print("Recommended Diet:")
        print(f"Calories to Maintain Weight: {recommendation['calories_to_maintain_weight']}")
        if diet_preference.lower() == "vegan":
            print(f"Recipe: {recommendation['Vegan_Recipe']}")
            print(f"Protein(g): {recommendation['V_Protein(g)']}")
            print(f"Fat(g): {recommendation['V_Fat(g)']}")
            print(f"Carbs(g): {recommendation['V_Carbs(g)']}")
        elif diet_preference.lower() == "mediterranean":
            print(f"Recipe: {recommendation['Mediterranean_Recipe']}")
            print(f"Protein(g): {recommendation['M_Protein(g)']}")
            print(f"Fat(g): {recommendation['M_Fat(g)']}")
            print(f"Carbs(g): {recommendation['M_Carbs(g)']}")
        print(f"BMI: {calculated_BMI:.2f}")
    else:
        print("No recommendation found.")
except ValueError as e:
    print(str(e))

Recommended Diet:
Calories to Maintain Weight: 2149.8144
Recipe: Vegan Chocolate-Ganache Frosting
Protein(g): 24.8
Fat(g): 133.13
Carbs(g): 160.68
BMI: 36.60


###Evaluation Metrics

In [ ]:
#evaluation metrics
from sklearn.metrics import precision_score, recall_score, f1_score

# Example lists of expected and predicted recommendations
expected_recommendations = ["Vegan Chocolate-Ganache Frosting", "Mediterranean Zucchini Salad", None, None]
predicted_recommendations = ["Vegan Chocolate-Ganache Frosting", "Mediterranean Zucchini Salad", None, "Middle Eastern Salad Bowls with Farro & Chicken"]

# Filter out None values and match recommendations by dietary preference
valid_expected = []
valid_predicted = []
for expected, predicted in zip(expected_recommendations, predicted_recommendations):
    if expected is not None and predicted is not None:
        valid_expected.append(expected)
        valid_predicted.append(predicted)

# Calculate precision, recall, and F1 score if there are valid recommendations
if valid_expected and valid_predicted:
    precision = precision_score(valid_expected, valid_predicted, average='weighted', zero_division=0)
    recall = recall_score(valid_expected, valid_predicted, average='weighted', zero_division=0)
    f1 = f1_score(valid_expected, valid_predicted, average='weighted', zero_division=0)

    # Print evaluation metrics
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")
else:
    print("No valid recommendations to evaluate.")

Precision: 1.00
Recall: 1.00
F1 Score: 1.00
